# Car Price Prediction — MVP Prototype

**Course project:** ML Systems / BANA 7075  
**Prototype goal:** Build a reproducible MVP for predicting vehicle selling price from the provided cleaned CSV.

### What this notebook implements

1. **Data pipeline**
   - Batch ingestion from the cleaned CSV
   - Data validation and quality checks
   - Reproducible sampling/splitting
   - Data version fingerprint using SHA-256
   - Processed-data storage as Parquet

2. **ML model development**
   - Ridge Regression baseline
   - Lasso Regression baseline
   - Random Forest Regressor
   - XGBoost Regressor
   - Common evaluation metrics: MAE, RMSE, R²
   - MLflow experiment tracking
   - Model artifact logging
   - Model registry entry for the selected MVP model

3. **Basic deployment preparation**
   - Save the complete preprocessing + model pipeline
   - Generate a prediction example
   - Prepare the artifact needed by an optional Streamlit/FastAPI demo

> **Important:** The CSV is already cleaned, so this prototype focuses on **validation, reproducibility, feature preparation, model experimentation, and tracking** rather than performing extensive data cleaning.


## 0. System Architecture

```text
Clean CSV
   │
   ▼
Batch Data Ingestion
   │
   ├──► SHA-256 Data Version Fingerprint
   │
   ▼
Data Validation
   │
   ▼
Feature Engineering
   │
   ├── Numeric features
   └── Categorical features → One-Hot Encoding
   │
   ▼
Train / Test Split
   │
   ├──────────────┬───────────────┬───────────────┐
   ▼              ▼               ▼               ▼
Ridge           Lasso       Random Forest      XGBoost
   │              │               │               │
   └──────────────┴───────────────┴───────────────┘
                          │
                          ▼
                    MLflow Tracking
              params + metrics + artifacts
                          │
                          ▼
                  Model Comparison
                          │
                          ▼
                 Register MVP Model
                          │
                          ▼
             Optional API / Streamlit App
```

This architecture directly maps to the assignment requirements: ingestion → validation → feature engineering → experimentation → model tracking/versioning → deployment preparation.


In [1]:
# ============================================================
# 1. INSTALL / IMPORT DEPENDENCIES
# ============================================================
# Run this cell once in JupyterLab.
#
# MLflow is installed here so the notebook can track experiments.
# XGBoost is used for the ensemble-model experiment.
#
# If your environment already has these packages, the command
# will simply confirm/update them.
# ============================================================

%pip install -q pandas numpy scikit-learn xgboost mlflow joblib pyarrow

import os
import json
import hashlib
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
print("Dependencies loaded successfully.")


Note: you may need to restart the kernel to use updated packages.


c:\Users\rjxl1\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies loaded successfully.


## 2. Project Configuration

The configuration below keeps the prototype reproducible. The full CSV contains hundreds of thousands of records, so the MVP uses a configurable sample for faster experimentation in JupyterLab.

You can increase `MVP_SAMPLE_SIZE` later if your computer can handle the larger models.

**Recommended first run:** 50,000 rows.  
**For a stronger final demo:** increase to 100,000+ rows if runtime permits.


In [2]:
# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

# Update this path only if your CSV is stored somewhere else.
DATA_PATH = Path("National_data_2_locations_Clean.csv")

# Output folders
ARTIFACT_DIR = Path("artifacts")
MODEL_DIR = Path("models")
PROCESSED_DIR = Path("data")
MLFLOW_DB = "sqlite:///mlflow.db"

# Reproducibility
RANDOM_STATE = 42

# MVP training size
MVP_SAMPLE_SIZE = 50_000

# Target variable
TARGET = "Selling Price"

# Create output directories
ARTIFACT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

print(f"Data path: {DATA_PATH.resolve()}")
print(f"Target: {TARGET}")
print(f"MVP sample size: {MVP_SAMPLE_SIZE:,}")


Data path: C:\Users\rjxl1\OneDrive\Documents\Masters Stuff\Fall 26'\BANA 7075\used-car-price-prediction\National_data_2_locations_Clean.csv
Target: Selling Price
MVP sample size: 50,000


## 3. Batch Data Ingestion

This is the **data ingestion stage** of the pipeline.

The prototype reads the cleaned CSV as a batch source and records basic metadata such as row count, column count, and file size.


In [3]:
# ============================================================
# 3. BATCH DATA INGESTION
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. "
        "Place the cleaned CSV in the same directory as this notebook "
        "or update DATA_PATH in the configuration cell."
    )

df = pd.read_csv(DATA_PATH)

print("Batch ingestion complete.")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"File size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

display(df.head())


Batch ingestion complete.
Rows: 459,343
Columns: 21
File size: 70.36 MB


,Year,Make,Model,Trim,Body,Transmission,Vehicle Identification Number,State Code,Odometer,Color,...,Seller,Market Price,Selling Price,Sale Date,Sale Year,Sale Month,Sale Month Name,Sale Day of Week,Sale Day Name,State
0,1999,Pontiac,Grand Prix,GTP,Sedan,Automatic,1g2wr5211xf210458,AZ,261949,Red,...,Pls Loan Store/Phoenix,50,300,2/25/2015,2015,2,February,2,Wednesday,Arizona
1,2001,Chrysler,Sebring,LX,Convertible,Automatic,1c3el45u71n673716,CA,223707,Blue,...,Purple Heart Services Inc,25,300,12/30/2014,2014,12,December,1,Tuesday,California
2,2000,Nissan,Maxima,SE,Sedan,Automatic,jn1ca31d8yt763569,FL,229381,Black,...,Courtesy Nissan Of Tampa,25,300,5/21/2015,2015,5,May,3,Thursday,Florida
3,1997,Nissan,Maxima,GLE,Sedan,Automatic,jn1ca21d3vt870132,FL,220607,Black,...,Autonation Nissan Orange Park,25,300,1/27/2015,2015,1,January,1,Tuesday,Florida
4,2002,Cadillac,Seville,SLS,sedan,Automatic,1g6ks54y82u199426,FL,255368,Red,...,Autonation Nissan Brandon,50,300,6/16/2015,2015,6,June,1,Tuesday,Florida


## 4. Data Versioning / Reproducibility

Because the source CSV is already cleaned, the most important MVP requirement is being able to prove **which exact input data produced the experiment**.

We create a SHA-256 hash of the CSV. If the file changes by even one byte, the hash changes.

The metadata JSON records:
- source filename
- SHA-256 hash
- row/column counts
- target variable
- random seed
- prototype sample size

This gives the experiment a reproducible data fingerprint. For a production workflow, the same file can also be placed under **DVC** or another dedicated data-versioning system.


In [4]:
# ============================================================
# 4. DATA VERSION FINGERPRINT
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    """Return SHA-256 hash for a file without loading the entire file into memory."""
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha.update(chunk)
    return sha.hexdigest()

data_hash = sha256_file(DATA_PATH)

data_version = {
    "source_file": DATA_PATH.name,
    "sha256": data_hash,
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "mvp_sample_size": MVP_SAMPLE_SIZE,
}

with open(ARTIFACT_DIR / "data_version.json", "w") as f:
    json.dump(data_version, f, indent=4)

print("Data version fingerprint:")
print(data_hash)
print("\nMetadata saved to:", ARTIFACT_DIR / "data_version.json")


Data version fingerprint:
f6c1f0fe3b580fa8387e8c5c7927f0bce816cd71f45a10546655bf37ee630379

Metadata saved to: artifacts\data_version.json


### Optional DVC command

If your group is using Git, you can add the CSV to DVC as an additional formal data-versioning layer:

```bash
pip install dvc
dvc init
dvc add National_data_2_locations_Clean.csv
git add National_data_2_locations_Clean.csv.dvc .gitignore
git commit -m "Version cleaned car price dataset"
```

For the recorded demo, you can explain that the **SHA-256 fingerprint is implemented in the MVP**, while DVC can be used to maintain dataset versions across group members and Git commits.


## 5. Data Validation

Even though the CSV was supplied as cleaned data, the ML pipeline should still validate its input before training.

The checks below verify:
- required columns exist
- there are no missing values
- the target is numeric
- selling prices are non-negative
- odometer values are non-negative
- year values are plausible
- categorical fields contain usable values
- duplicate rows are reported

The validation report is saved as an MLflow artifact later.


In [5]:
# ============================================================
# 5. DATA VALIDATION
# ============================================================

required_columns = [
    "Year", "Make", "Model", "Trim", "Body", "Transmission",
    "Vehicle Identification Number", "State Code", "Odometer",
    "Color", "Interior", "Seller", "Market Price", "Selling Price",
    "Sale Date", "Sale Year", "Sale Month", "Sale Month Name",
    "Sale Day of Week", "Sale Day Name", "State"
]

validation = {
    "required_columns_present": set(required_columns).issubset(df.columns),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "negative_selling_prices": int((df[TARGET] < 0).sum()),
    "negative_odometer": int((df["Odometer"] < 0).sum()),
    "invalid_vehicle_years": int(((df["Year"] < 1900) | (df["Year"] > 2026)).sum()),
    "non_numeric_target": not pd.api.types.is_numeric_dtype(df[TARGET]),
}

print("Validation results:")
for check, result in validation.items():
    print(f"  {check}: {result}")

# Hard-fail only on conditions that make model training unsafe.
hard_fail_checks = {
    "required_columns_present": validation["required_columns_present"],
    "no_missing_values": validation["missing_values"] == 0,
    "non_negative_target": validation["negative_selling_prices"] == 0,
    "non_negative_odometer": validation["negative_odometer"] == 0,
    "numeric_target": validation["non_numeric_target"],
}

# Note: the 'numeric_target' check above is intentionally named for readability.
# A numeric target should make this True; therefore check the dtype directly.
hard_fail_checks["numeric_target"] = pd.api.types.is_numeric_dtype(df[TARGET])

failed = [name for name, passed in hard_fail_checks.items() if not passed]

if failed:
    raise ValueError(f"Data validation failed: {failed}")

with open(ARTIFACT_DIR / "validation_report.json", "w") as f:
    json.dump(validation, f, indent=4)

print("\nAll required validation checks passed.")


Validation results:
  required_columns_present: True
  missing_values: 0
  duplicate_rows: 0
  negative_selling_prices: 0
  negative_odometer: 0
  invalid_vehicle_years: 0
  non_numeric_target: False

All required validation checks passed.


## 6. Select the MVP Modeling Dataset

### Target
`Selling Price`

### Features
We intentionally exclude:
- **Vehicle Identification Number** — effectively unique for each row and not useful for generalization.
- **Sale Date** — replaced by the already available sale-time fields.
- **Sale Month Name / Sale Day Name** — redundant with numeric month/day fields.
- **State Code** — redundant with State.
- **Seller** — very high-cardinality field; excluding it keeps the MVP computationally manageable and reduces the risk of memorizing individual sellers.

We also create **Vehicle Age**, which represents the age of the vehicle at sale time.

`Market Price` is retained because it is an available vehicle valuation field. In the presentation, explicitly mention that the team should evaluate whether this feature would be available at prediction time in the intended real-world deployment scenario.


In [6]:
# ============================================================
# 6. FEATURE SELECTION + ENGINEERING
# ============================================================

model_df = df.copy()

# Feature engineering: vehicle age at the time of sale.
model_df["Vehicle Age"] = model_df["Sale Year"] - model_df["Year"]

numeric_features = [
    "Year",
    "Odometer",
    "Market Price",
    "Sale Year",
    "Sale Month",
    "Sale Day of Week",
    "Vehicle Age",
]

categorical_features = [
    "Make",
    "Model",
    "Trim",
    "Body",
    "Transmission",
    "Color",
    "Interior",
    "State",
]

feature_columns = numeric_features + categorical_features

X = model_df[feature_columns]
y = model_df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Store feature documentation for reproducibility.
feature_info = {
    "target": TARGET,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "excluded_columns": [
        "Vehicle Identification Number",
        "Sale Date",
        "Sale Month Name",
        "Sale Day Name",
        "State Code",
        "Seller"
    ],
}

with open(ARTIFACT_DIR / "feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=4)


Feature matrix shape: (459343, 15)
Target shape: (459343,)

Numeric features: ['Year', 'Odometer', 'Market Price', 'Sale Year', 'Sale Month', 'Sale Day of Week', 'Vehicle Age']
Categorical features: ['Make', 'Model', 'Trim', 'Body', 'Transmission', 'Color', 'Interior', 'State']


## 7. Create a Reproducible MVP Sample and Train/Test Split

A fixed random seed ensures that every group member can reproduce the same sample and split.

We use:
- **80% training**
- **20% testing**
- `random_state = 42`


In [7]:
# ============================================================
# 7. SAMPLE + TRAIN/TEST SPLIT
# ============================================================

sample_n = min(MVP_SAMPLE_SIZE, len(model_df))

sample_df = model_df.sample(
    n=sample_n,
    random_state=RANDOM_STATE
).reset_index(drop=True)

X_sample = sample_df[feature_columns]
y_sample = sample_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Sample rows: {len(sample_df):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")

# Save the exact modeling sample so another group member can reproduce the demo.
sample_df.to_parquet(
    PROCESSED_DIR / "mvp_modeling_sample.parquet",
    index=False
)

print("\nProcessed MVP dataset saved to:")
print(PROCESSED_DIR / "mvp_modeling_sample.parquet")


Sample rows: 50,000
Training rows: 40,000
Testing rows: 10,000

Processed MVP dataset saved to:
data\mvp_modeling_sample.parquet


## 8. Build the Shared Preprocessing Pipeline

All models use the same preprocessing approach so that the comparison is fair.

- Numeric variables → standardization
- Categorical variables → one-hot encoding
- Unknown categories at inference time → ignored

The preprocessing is placed **inside each model pipeline**, which is important for deployment because the exact transformations used during training are saved together with the model.


In [8]:
# ============================================================
# 8. PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2
            ),
            categorical_features
        ),
    ],
    remainder="drop"
)

def make_pipeline(model):
    """Create a complete preprocessing + model pipeline."""
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

print("Shared preprocessing pipeline created.")


Shared preprocessing pipeline created.


# 9. MLflow Experiment Tracking Setup

MLflow will be used to record:

- model type
- hyperparameters
- training/test row counts
- MAE
- RMSE
- R²
- data version hash
- feature documentation
- validation report
- trained model artifact

The SQLite backend keeps the MLflow tracking database local to the project, which is convenient for a JupyterLab prototype.


In [9]:
# ============================================================
# 9. MLFLOW SETUP
# ============================================================

mlflow.set_tracking_uri(MLFLOW_DB)

EXPERIMENT_NAME = "Car_Price_Prediction_MVP"

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)


MLflow tracking URI: sqlite:///mlflow.db
Experiment: Car_Price_Prediction_MVP


## 10. Evaluation Helper

All models will be evaluated using the same metrics.

### MAE
Average absolute prediction error. Easier to interpret in dollars.

### RMSE
Penalizes large prediction errors more heavily than MAE.

### R²
Measures the proportion of target variance explained by the model.

For a car-price prediction system, **MAE and RMSE are especially useful** because the business user ultimately cares about how many dollars the prediction is off by.


In [10]:
# ============================================================
# 10. EVALUATION HELPER
# ============================================================

def evaluate_model(model, X_test, y_test):
    """Calculate common regression metrics."""
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }


# 11. Ridge Regression — Baseline Model

Ridge is the primary **linear baseline**.

The purpose of the baseline is not necessarily to be the final model. It gives the group a simple reference point so we can demonstrate whether the ensemble methods provide meaningful improvement.


In [11]:
# ============================================================
# 11. RIDGE REGRESSION BASELINE
# ============================================================

ridge_model = make_pipeline(
    Ridge(alpha=10.0)
)

with mlflow.start_run(run_name="Ridge_Baseline") as run:
    ridge_model.fit(X_train, y_train)
    ridge_metrics = evaluate_model(ridge_model, X_test, y_test)

    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 10.0)
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("data_sha256", data_hash)
    mlflow.log_param("random_state", RANDOM_STATE)

    for metric_name, metric_value in ridge_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    mlflow.log_artifact(str(ARTIFACT_DIR / "data_version.json"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "validation_report.json"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "feature_info.json"))

    mlflow.sklearn.log_model(ridge_model, "model")

    ridge_run_id = run.info.run_id

print("Ridge metrics:", ridge_metrics)
print("MLflow run ID:", ridge_run_id)


2026/10/05 10:23:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Ridge metrics: {'MAE': 1083.8366618435923, 'RMSE': 1754.7644242564106, 'R2': 0.9675088386302404}
MLflow run ID: fc49d21c8bf94f20bbafc3c9e56b68d3


# 12. Lasso Regression

Lasso provides a second linear baseline and introduces **L1 regularization**, which can shrink less useful coefficients toward zero.

This gives the group a meaningful comparison between two regularized linear approaches before moving to tree-based ensemble models.


In [12]:
# ============================================================
# 12. LASSO REGRESSION
# ============================================================

lasso_model = make_pipeline(
    Lasso(
        alpha=0.001,
        max_iter=5000,
        selection="random",
        random_state=RANDOM_STATE
    )
)

with mlflow.start_run(run_name="Lasso_Baseline") as run:
    lasso_model.fit(X_train, y_train)
    lasso_metrics = evaluate_model(lasso_model, X_test, y_test)

    mlflow.log_param("model_type", "Lasso Regression")
    mlflow.log_param("alpha", 0.001)
    mlflow.log_param("max_iter", 5000)
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("data_sha256", data_hash)
    mlflow.log_param("random_state", RANDOM_STATE)

    for metric_name, metric_value in lasso_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    mlflow.log_artifact(str(ARTIFACT_DIR / "data_version.json"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "validation_report.json"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "feature_info.json"))

    mlflow.sklearn.log_model(lasso_model, "model")

    lasso_run_id = run.info.run_id

print("Lasso metrics:", lasso_metrics)
print("MLflow run ID:", lasso_run_id)


2026/10/05 10:25:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Lasso metrics: {'MAE': 1105.6348647872603, 'RMSE': 1784.6660774495497, 'R2': 0.9663920880613447}
MLflow run ID: 4604e3175a374d5081c21ffaf9cc7c8f


# 13. Random Forest — Ensemble Experiment

Random Forest is the first ensemble model.

The MVP intentionally uses a moderate configuration so it can run in a student JupyterLab environment. After the first successful run, the group can tune:
- number of trees
- maximum tree depth
- minimum samples per leaf
- feature sampling

All changes should be logged as separate MLflow runs rather than overwriting the original experiment.

> **Revision note:** the first pass at these hyperparameters (`n_estimators=150`, `max_depth=18`, `max_features="sqrt"`) left Random Forest noticeably behind both the linear baselines and XGBoost on the same split. With high-cardinality one-hot categoricals (Make/Model/Trim) and a strongly linear predictor (`Market Price`) in the mix, a shallow `sqrt`-feature forest needs many more, deeper trees to rediscover that relationship the way Ridge gets "for free." The configuration below increases `n_estimators`, removes the depth cap (bounded only by `min_samples_leaf`), and widens `max_features` so each split has a better chance of finding `Market Price` or `Vehicle Age`.


In [13]:
# ============================================================
# 13. RANDOM FOREST EXPERIMENT (revised hyperparameters)
# ============================================================
rf_params = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_leaf": 2,
    "max_features": 0.5,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

rf_model = make_pipeline(
    RandomForestRegressor(**rf_params)
)

with mlflow.start_run(run_name="RandomForest_Ensemble_v2") as run:
    rf_model.fit(X_train, y_train)

    rf_metrics = evaluate_model(
        rf_model,
        X_test,
        y_test
    )

    # Log model parameters
    mlflow.log_params({
        "model_type": "Random Forest",
        **rf_params,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "data_sha256": data_hash,
    })

    # Log evaluation metrics
    for metric_name, metric_value in rf_metrics.items():
        mlflow.log_metric(
            metric_name,
            metric_value
        )

    # Log supporting artifacts
    mlflow.log_artifact(
        str(ARTIFACT_DIR / "data_version.json")
    )

    mlflow.log_artifact(
        str(ARTIFACT_DIR / "validation_report.json")
    )

    mlflow.log_artifact(
        str(ARTIFACT_DIR / "feature_info.json")
    )

    # --------------------------------------------------------
    # MLflow model logging
    #
    # Random Forest contains sklearn.tree._tree.Tree objects.
    # We explicitly trust this exact type because the model was
    # trained locally from our own trusted source code/data.
    # --------------------------------------------------------
    mlflow.sklearn.log_model(
        rf_model,
        "model",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree"
        ]
    )

    rf_run_id = run.info.run_id

print("Random Forest metrics:", rf_metrics)
print("MLflow run ID:", rf_run_id)

KeyboardInterrupt: 

# 14. XGBoost — Ensemble Experiment

XGBoost is a gradient-boosted tree ensemble.

This experiment tests whether boosting can capture nonlinear relationships between:
- vehicle age
- mileage
- market price
- make/model/trim
- vehicle characteristics
- geographic information

The model is configured with histogram-based tree construction (`hist`) to improve training speed.


In [ ]:
# ============================================================
# 14. XGBOOST EXPERIMENT
# ============================================================

xgb_params = {
    "n_estimators": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

xgb_model = make_pipeline(
    XGBRegressor(**xgb_params)
)

with mlflow.start_run(run_name="XGBoost_Ensemble") as run:

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    xgb_model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Evaluate model
    # --------------------------------------------------------
    xgb_metrics = evaluate_model(
        xgb_model,
        X_test,
        y_test
    )

    # --------------------------------------------------------
    # Log hyperparameters
    # --------------------------------------------------------
    mlflow.log_params({
        "model_type": "XGBoost",
        **xgb_params,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "data_sha256": data_hash,
    })

    # --------------------------------------------------------
    # Log evaluation metrics
    # --------------------------------------------------------
    for metric_name, metric_value in xgb_metrics.items():
        mlflow.log_metric(
            metric_name,
            metric_value
        )

    # --------------------------------------------------------
    # Log supporting artifacts
    # --------------------------------------------------------
    mlflow.log_artifact(
        str(ARTIFACT_DIR / "data_version.json")
    )

    mlflow.log_artifact(
        str(ARTIFACT_DIR / "validation_report.json")
    )

    mlflow.log_artifact(
        str(ARTIFACT_DIR / "feature_info.json")
    )

    # --------------------------------------------------------
    # Log XGBoost model to MLflow
    #
    # MLflow/skops identifies these XGBoost classes as
    # untrusted during serialization:
    #
    #   xgboost.core.Booster
    #   xgboost.sklearn.XGBRegressor
    #
    # We explicitly trust ONLY those two types.
    # --------------------------------------------------------
    mlflow.sklearn.log_model(
        xgb_model,
        "model",
        skops_trusted_types=[
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBRegressor"
        ]
    )

    xgb_run_id = run.info.run_id

print("XGBoost metrics:", xgb_metrics)
print("MLflow run ID:", xgb_run_id)


2026/09/22 13:42:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost metrics: {'MAE': 1106.549072265625, 'RMSE': 2074.341100205075, 'R2': 0.9545966386795044}
MLflow run ID: 47db976684d4498d91ec61dbb5c6a810


# 15. Compare All Experiments

This table provides the evidence for the model-development portion of the assignment.

The comparison is based on the same test set. For regression, lower MAE/RMSE indicates smaller prediction error, while higher R² indicates more variance explained.

**Do not choose a model solely from one metric.** In the final project, the team should also consider training time, model size, interpretability, and deployment requirements.


In [ ]:
# ============================================================
# 15. MODEL COMPARISON
# ============================================================

results = pd.DataFrame([
    {"Model": "Ridge Regression", **ridge_metrics, "Run ID": ridge_run_id},
    {"Model": "Lasso Regression", **lasso_metrics, "Run ID": lasso_run_id},
    {"Model": "Random Forest", **rf_metrics, "Run ID": rf_run_id},
    {"Model": "XGBoost", **xgb_metrics, "Run ID": xgb_run_id},
])

results = results.sort_values("RMSE").reset_index(drop=True)

display(results)

results.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)

print("Comparison saved to:", ARTIFACT_DIR / "model_comparison.csv")


,Model,MAE,RMSE,R2,Run ID
0,Ridge Regression,1083.836662,1754.764424,0.967509,aa3fee6b3ce34141a1d9f40d1719cb1e
1,Lasso Regression,1105.634865,1784.666077,0.966392,790b23fe78264a3190f14d676cddbd32
2,Random Forest,1090.905529,1811.349726,0.965380,49b3bc83386c480e97f98f071b6f3d3c
3,XGBoost,1106.549072,2074.341100,0.954597,47db976684d4498d91ec61dbb5c6a810


Comparison saved to: artifacts\model_comparison.csv


# 16. Select and Save the MVP Model

For the prototype, the model with the lowest **test RMSE** is selected automatically.

This is an engineering rule for the MVP rather than a claim that RMSE is the only appropriate production criterion. The final project can add business constraints and additional validation.

The **complete pipeline** is saved, meaning preprocessing and the trained model remain together for future inference.


In [ ]:
# ============================================================
# 16. SELECT MVP MODEL + SAVE COMPLETE PIPELINE
# ============================================================

model_objects = {
    "Ridge Regression": ridge_model,
    "Lasso Regression": lasso_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
}

best_model_name = results.iloc[0]["Model"]
best_model = model_objects[best_model_name]
best_run_id = results.iloc[0]["Run ID"]

best_model_path = MODEL_DIR / "best_car_price_model.joblib"
joblib.dump(best_model, best_model_path)

print(f"Selected MVP model based on test RMSE: {best_model_name}")
print(f"Run ID: {best_run_id}")
print(f"Saved model: {best_model_path}")


Selected MVP model based on test RMSE: Ridge Regression
Run ID: aa3fee6b3ce34141a1d9f40d1719cb1e
Saved model: models\best_car_price_model.joblib


# 17. Model Versioning / Registry

The assignment asks for at least one model to be versioned.

The code below registers the selected MVP model in MLflow's Model Registry.

Every time the group selects a new MVP and reruns the registration step, MLflow can create another model version. The alias `champion` is used to identify the currently selected version without hard-coding a version number into the application.


In [ ]:
# ============================================================
# 17. REGISTER MODEL VERSION IN MLFLOW
# ============================================================

MODEL_REGISTRY_NAME = "CarPricePrediction"

client = mlflow.tracking.MlflowClient()

# The best model was already logged under its original run.
# Register that logged model as a version in the model registry.
model_uri = f"runs:/{best_run_id}/model"

try:
    registered = mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_REGISTRY_NAME
    )

    version_number = registered.version

    # MLflow aliases provide a current-model pointer.
    client.set_registered_model_alias(
        MODEL_REGISTRY_NAME,
        "champion",
        version_number
    )

    print(f"Registered model: {MODEL_REGISTRY_NAME}")
    print(f"Version: {version_number}")
    print("Alias: champion")

except Exception as e:
    print("Model registry step did not complete.")
    print("If your MLflow version/backend does not support local registry storage,")
    print("the model is still versioned by its MLflow run ID and saved artifact.")
    print("Details:", e)


Registered model 'CarPricePrediction' already exists. Creating a new version of this model...
2026/09/22 13:42:42 WARNING mlflow.tracking._model_registry.fluent: Run with id aa3fee6b3ce34141a1d9f40d1719cb1e has no artifacts at artifact path 'model', registering model based on models:/m-f3bc95ef76b846d988602d57a34db17e instead


Registered model: CarPricePrediction
Version: 2
Alias: champion


Created version '2' of model 'CarPricePrediction'.


# 18. Verify Inference

A model is not complete from a systems perspective until we show that the saved artifact can load and make a prediction.

This cell loads the saved pipeline from disk and performs inference on one test record.


In [ ]:
# ============================================================
# 18. LOAD SAVED MODEL + TEST INFERENCE
# ============================================================

loaded_model = joblib.load(best_model_path)

example_input = X_test.iloc[[0]]
actual_price = float(y_test.iloc[0])
predicted_price = float(loaded_model.predict(example_input)[0])

print(f"Actual selling price:    ${actual_price:,.2f}")
print(f"Predicted selling price: ${predicted_price:,.2f}")
print(f"Absolute error:          ${abs(actual_price - predicted_price):,.2f}")


Actual selling price:    $4,800.00
Predicted selling price: $4,308.49
Absolute error:          $491.51


## 18b. Market Price Ablation Study — With vs. Without `Market Price`

The proposal describes `Market Price` (our MMR-style benchmark) as something to **compare our predictions against**, not necessarily something the model should be allowed to see as an input — using a professional appraisal value to predict the final sale price risks leaking most of the answer into the model, since MMR is already a strong estimate of what the car will sell for.

To make that tradeoff explicit rather than just noting it as a caveat, this section retrains the strongest model type from the comparison above **without** `Market Price` in the feature set, and reports both results side by side. This gives us:

- an **MMR-assisted model** (Section 16's selection) — best case, assumes a market-price estimate is available at prediction time
- a **raw-features model** — trained only on vehicle attributes (year, mileage, make/model/trim, body, transmission, color, interior, state, sale timing) — the more honest picture of what the system can do if MMR isn't available (e.g., for a private seller pricing a car from scratch)

Both runs are logged to MLflow under distinct run names so the comparison is reproducible.


In [ ]:
# ============================================================
# 18b. MARKET PRICE ABLATION — WITH vs. WITHOUT Market Price
# ============================================================

# Re-use the winning model TYPE from the comparison table (Section 15/16),
# but retrain it on a feature set that excludes Market Price.

no_mp_numeric_features = [f for f in numeric_features if f != "Market Price"]
no_mp_feature_columns = no_mp_numeric_features + categorical_features

X_train_no_mp = X_train[no_mp_feature_columns]
X_test_no_mp = X_test[no_mp_feature_columns]

no_mp_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), no_mp_numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", min_frequency=2),
            categorical_features,
        ),
    ],
    remainder="drop",
)


def make_no_mp_pipeline(model):
    return Pipeline(steps=[("preprocessor", no_mp_preprocessor), ("model", model)])


# Map the winning model name back to a fresh, unfitted estimator of the same type/params.
winning_estimator_factory = {
    "Ridge Regression": lambda: Ridge(alpha=10.0),
    "Lasso Regression": lambda: Lasso(
        alpha=0.001, max_iter=5000, selection="random", random_state=RANDOM_STATE
    ),
    "Random Forest": lambda: RandomForestRegressor(**rf_params),
    "XGBoost": lambda: XGBRegressor(**xgb_params),
}

no_mp_model = make_no_mp_pipeline(winning_estimator_factory[best_model_name]())

with mlflow.start_run(run_name=f"{best_model_name.replace(' ', '_')}_No_MarketPrice") as run:
    no_mp_model.fit(X_train_no_mp, y_train)
    no_mp_metrics = evaluate_model(no_mp_model, X_test_no_mp, y_test)

    mlflow.log_param("model_type", f"{best_model_name} (no Market Price)")
    mlflow.log_param("excluded_feature", "Market Price")
    mlflow.log_param("train_rows", len(X_train_no_mp))
    mlflow.log_param("test_rows", len(X_test_no_mp))
    mlflow.log_param("data_sha256", data_hash)

    for metric_name, metric_value in no_mp_metrics.items():
        mlflow.log_metric(metric_name, metric_value)

    no_mp_run_id = run.info.run_id

ablation_results = pd.DataFrame([
    {"Model": f"{best_model_name} (with Market Price)", **results.iloc[0][["MAE", "RMSE", "R2"]].to_dict(), "Run ID": best_run_id},
    {"Model": f"{best_model_name} (without Market Price)", **no_mp_metrics, "Run ID": no_mp_run_id},
])

display(ablation_results)

ablation_results.to_csv(ARTIFACT_DIR / "market_price_ablation.csv", index=False)
print("Ablation comparison saved to:", ARTIFACT_DIR / "market_price_ablation.csv")
print(
    "\nInterpretation: a large gap between the two rows means Market Price is doing most of the "
    "work, and the 'without' row is the more honest estimate of real-world performance when no "
    "prior market valuation is available at prediction time."
)

,Model,MAE,RMSE,R2,Run ID
0,Ridge Regression (with Market Price),1083.836662,1754.764424,0.967509,aa3fee6b3ce34141a1d9f40d1719cb1e
1,Ridge Regression (without Market Price),2372.904781,4199.523567,0.813908,43623980bc8e4c3683d146d2fb0cd70d


Ablation comparison saved to: artifacts\market_price_ablation.csv

Interpretation: a large gap between the two rows means Market Price is doing most of the work, and the 'without' row is the more honest estimate of real-world performance when no prior market valuation is available at prediction time.
